In [ ]:
!pip install opendatasets --quiet
!pip install torchsummary --quiet

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
data_df = pd.read_csv('/content/rice-type-classification/riceClassification.csv')
data_df.head()

In [ ]:
data_df.dropna(inplace=True)
data_df.drop(['id'],axis=1,inplace=True)
print(data_df.shape)
print(data_df.columns)
original_df = data_df.copy()

In [ ]:
data_df.head()

In [ ]:
for columns in data_df.columns:
  data_df[columns] = data_df[columns]/data_df[columns].abs().max()

data_df.head()

In [ ]:
X = np.array(data_df.iloc[:, :-1])

In [ ]:
y = np.array(data_df.iloc[:,-1])

In [ ]:
print(X.shape,y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [ ]:
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5,random_state=42)

In [ ]:
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)
print(X_val.shape,y_val.shape)

In [ ]:
class dataset(Dataset):
  def __init__(self,X,y):
    self.X = torch.tensor(X,dtype=torch.float32).to(device)
    self.y = torch.tensor(y,dtype=torch.float32).to(device)

  def __len__(self):
    return len(self.X)

  def __getitem__(self,idx):
    return self.X[idx],self.y[idx]

In [ ]:
training_data = dataset(X_train,y_train)
validation_data = dataset(X_val,y_val)
testing_data = dataset(X_test,y_test)

In [ ]:
training_dataloader = DataLoader(training_data,batch_size=8,shuffle=True)
validation_dataloader = DataLoader(validation_data,batch_size=8,shuffle=True)
testing_dataloader = DataLoader(testing_data,batch_size=8,shuffle=False)

In [ ]:
for x,y in training_dataloader:
  print(x)
  print("======")
  print(y)
  break

In [ ]:
HIDDEN_NEURONS = 10
class MyModel(nn.Module):
  def __init__(self):
    super(MyModel,self).__init__()
    self.input_layer = nn.Linear(X.shape[1],HIDDEN_NEURONS)
    self.hidden_layer = nn.Linear(HIDDEN_NEURONS,1)
    self.sigmiod = nn.Sigmoid()

  def forward(self,x):
    x = self.input_layer(x)
    x = self.hidden_layer(x)
    x = self.sigmiod(x)
    return x
model = MyModel().to(device)

In [ ]:
summary(model,(x.shape[1],))

In [ ]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(),lr=0.001)

In [ ]:
total_loss_train_plot = []
total_loss_val_plot = []
total_acc_train_plot = []
total_acc_val_plot = []

epochs = 10
for i in range(epochs):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_val = 0
  total_loss_val = 0

  for data in training_dataloader:
    inputs, labels = data
    prediction = model(inputs).squeeze(1)
    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()

    # Calculate accuracy as sum of correct predictions per batch
    acc = (prediction.round() == labels).sum().item()
    total_acc_train += acc
    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for data in validation_dataloader:
      inputs, labels = data
      prediction = model(inputs).squeeze(1)
      batch_loss = criterion(prediction,labels)
      total_loss_val += batch_loss.item()
      # Calculate accuracy as sum of correct predictions per batch
      acc = (prediction.round() == labels).sum().item()
      total_acc_val += acc

  # Normalize loss by number of batches and accuracy by total number of samples
  avg_train_loss = total_loss_train / len(training_dataloader)
  avg_val_loss = total_loss_val / len(validation_dataloader)
  overall_train_acc = total_acc_train / training_data.__len__() * 100
  overall_val_acc = total_acc_val / validation_data.__len__() * 100

  total_loss_train_plot.append(round(avg_train_loss, 4))
  total_loss_val_plot.append(round(avg_val_loss, 4))
  total_acc_train_plot.append(round(overall_train_acc, 4))
  total_acc_val_plot.append(round(overall_val_acc, 4))

  print(f"Epoch {i} Training Loss {avg_train_loss:.4f} Validation Loss {avg_val_loss:.4f}")
  print(f"Epoch {i} Training Acc {overall_train_acc:.2f}% Validation Acc {overall_val_acc:.2f}%")


In [ ]:
with torch.no_grad():
  total_acc_test = 0
  total_loss_test = 0
  for data in testing_dataloader:
    inputs, labels = data
    prediction = model(inputs).squeeze(1)
    batch_loss = criterion(prediction,labels)
    acc = ((prediction).round() == labels).sum().item()

    total_acc_test += acc
    total_loss_test += batch_loss.item()

  print("Test Loss", total_loss_test, "Test Acc",((total_acc_test/X_test.shape[0])*100) )

In [ ]:
with torch.no_grad():
  total_loss_test = 0
  total_acc_test = 0
  for data in testing_dataloader:
    inputs, labels = data

    prediction = model(inputs).squeeze(1)

    batch_loss_test = criterion((prediction), labels)
    total_loss_test += batch_loss_test.item()
    acc = ((prediction).round() == labels).sum().item()
    total_acc_test += acc

print(f"Accuracy Score is: {round((total_acc_test/X_test.shape[0])*100, 2)}%")

In [ ]:
plt.plot(total_loss_train_plot, label='Training Loss')
plt.plot(total_loss_val_plot, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.plot(total_acc_train_plot, label='Training Accuracy')
plt.plot(total_acc_val_plot, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
input("Area: ")

In [ ]:
area = float(input("Area: "))/original_df['Area'].abs().max()
MajorAxisLength = float(input("Major Axis Length: "))/original_df['MajorAxisLength'].abs().max()
MinorAxisLength = float(input("Minor Axis Length: "))/original_df['MinorAxisLength'].abs().max()
Eccentricity = float(input("Eccentricity: "))/original_df['Eccentricity'].abs().max()
ConvexArea = float(input("Convex Area: "))/original_df['ConvexArea'].abs().max()
EquivDiameter = float(input("EquivDiameter: "))/original_df['EquivDiameter'].abs().max()
Extent = float(input("Extent: "))/original_df['Extent'].abs().max()
Perimeter = float(input("Perimeter: "))/original_df['Perimeter'].abs().max()
Roundness = float(input("Roundness: "))/original_df['Roundness'].abs().max()
AspectRation = float(input("AspectRation: "))/original_df['AspectRation'].abs().max()

In [ ]:
my_inputs = [area, MajorAxisLength, MinorAxisLength, Eccentricity, ConvexArea, EquivDiameter, Extent, Perimeter, Roundness, AspectRation]

print("="*20)
model_inputs = torch.Tensor(my_inputs).to(device)
prediction = (model(model_inputs))
print(prediction)
print("Class is: ", round(prediction.item()))